In [ ]:
import sys
sys.path.append("/exp/sbnd/data/users/lynnt/xsection/")

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import nueana as nue
from unfolding.wienersvd import WienerSVD

%load_ext autoreload
%autoreload 2

In [ ]:
# Update these paths for your local installation
checkpoint_dir = "/exp/sbnd/data/users/lynnt/xsection/notebooks/june2026/"
dfs_dir        = "/exp/sbnd/data/users/lynnt/xsection/samples/MCP2025B_v10_06_00_09/dfs_nu26/"

## load checkpoints

In [ ]:
# sideband_outputs: dict keyed by var_save_name ('energy', 'direction'),
# each a SystematicsOutput for the control (sideband) region.
mcmc_sideband_df, sideband_outputs = nue.load_sideband_checkpoint(
    checkpoint_dir + "sideband_checkpoint.pkl"
)

# signal_outputs_total: full selected region (signal + background)
# signal_outputs_sig:   signal component only (signal == 0)
# signal_outputs_bkg:   background component only (signal != 0)
mcmc_signal_df, mcnue_sig_df, signal_outputs_total, signal_outputs_sig, signal_outputs_bkg = \
    nue.load_signal_checkpoint(checkpoint_dir + "signal_checkpoint.pkl")

var_energy    = nue.electron_energy()
var_direction = nue.electron_direction()
mcbnb_pot     = signal_outputs_total[var_energy.var_save_name].mcbnb_pot

## CCBC block structure

In [ ]:
# The CCBC covariance has a block structure over (B_S, P_S, n_C): the background
# and signal components in the signal region, and the total rate in the control region.
# plot_ccbc_blocks visualises the full block matrix and its fractional/correlation form.
for var_name, var_cfg in [('energy', var_energy), ('direction', var_direction)]:
    nue.plot_ccbc_blocks(
        Bs_output    = signal_outputs_bkg[var_name],
        nc_output    = sideband_outputs[var_name],
        allowed_keys = ['GENIE', 'Flux', 'Geant4'],
        var          = var_name,
    )
    plt.suptitle(f"GENIE + Flux + Geant4 — {var_cfg.var_plot_name}")
    plt.show()

## Constraint comparison by systematic type

In [ ]:
# get_ccbc_cov assembles the block covariance matrices and applies the sideband
# linear constraint:  B_S_constrained = B_S_CV + C_{B_S,n_C} C_{n_C,n_C}^{-1} (n_C - n_C^CV)
# Running once per systematic type shows which groups benefit most from the constraint.
syst_types = ['GENIE', 'Flux', 'Geant4', 'DetVar']

energy_covs = [
    nue.get_ccbc_cov(
        Bs_output    = signal_outputs_bkg['energy'],
        Ps_output    = signal_outputs_sig['energy'],
        nc_output    = sideband_outputs['energy'],
        ns_output    = signal_outputs_total['energy'],
        allowed_keys = ['MCstat', syst],
    )
    for syst in syst_types
]
energy_cov_mcstat = nue.get_ccbc_cov(
    Bs_output    = signal_outputs_bkg['energy'],
    Ps_output    = signal_outputs_sig['energy'],
    nc_output    = sideband_outputs['energy'],
    ns_output    = signal_outputs_total['energy'],
    allowed_keys = ['MCstat'],
)

output = nue.plot_ccbc_constraint(
    cov_list  = energy_covs,
    cov_stat  = energy_cov_mcstat,
    Bs_hist   = signal_outputs_bkg['energy'].rate_hist_cv,
    ns_hist   = signal_outputs_total['energy'].rate_hist_cv,
    var_config= var_energy,
    list_keys = syst_types,
)
plt.show()

In [ ]:
direct_covs = [
    nue.get_ccbc_cov(
        Bs_output    = signal_outputs_bkg['direction'],
        Ps_output    = signal_outputs_sig['direction'],
        nc_output    = sideband_outputs['direction'],
        allowed_keys = ['MCstat', syst],
    )
    for syst in syst_types
]
direct_cov_mcstat = nue.get_ccbc_cov(
    Bs_output    = signal_outputs_bkg['direction'],
    Ps_output    = signal_outputs_sig['direction'],
    nc_output    = sideband_outputs['direction'],
    allowed_keys = ['MCstat'],
)

output = nue.plot_ccbc_constraint(
    cov_list  = direct_covs,
    cov_stat  = direct_cov_mcstat,
    Bs_hist   = signal_outputs_bkg['direction'].rate_hist_cv,
    ns_hist   = signal_outputs_total['direction'].rate_hist_cv,
    var_config= var_direction,
    list_keys = syst_types,
)
plt.show()

## Total constraint summary

In [ ]:
# Combined GENIE + Flux + Geant4 covariance; plot_ccbc_summary shows pre- and
# post-constraint uncertainty on the background and total signal-region rate.
# The norm percentages use the norm_cov_* scalar fields (variable-independent).
energy_tot_cov = nue.get_ccbc_cov(
    Bs_output    = signal_outputs_bkg['energy'],
    Ps_output    = signal_outputs_sig['energy'],
    nc_output    = sideband_outputs['energy'],
    ns_output    = signal_outputs_total['energy'],
    allowed_keys = ['MCstat', 'GENIE', 'Flux', 'Geant4'],
)
direct_tot_cov = nue.get_ccbc_cov(
    Bs_output    = signal_outputs_bkg['direction'],
    Ps_output    = signal_outputs_sig['direction'],
    nc_output    = sideband_outputs['direction'],
    allowed_keys = ['MCstat', 'GENIE', 'Flux', 'Geant4'],
)

nue.plot_ccbc_summary(
    ccbc_cov  = energy_tot_cov,
    cov_stat  = energy_cov_mcstat,
    Bs_hist   = signal_outputs_bkg['energy'].rate_hist_cv,
    ns_hist   = signal_outputs_total['energy'].rate_hist_cv,
    var_config= var_energy,
)
plt.show()

nue.plot_ccbc_summary(
    ccbc_cov  = direct_tot_cov,
    cov_stat  = direct_cov_mcstat,
    Bs_hist   = signal_outputs_bkg['direction'].rate_hist_cv,
    ns_hist   = signal_outputs_total['direction'].rate_hist_cv,
    var_config= var_direction,
)
plt.show()

## Fake data tests (with CCBC)

In [ ]:
# UnfoldInput bundles the response matrix and systematic covariances.
# cv_signal is stored in absolute event-count units at mcbnb_pot (weights_mc summed).
# The default xsec_scale=1/(integrated_flux*NTARGETS) converts to cross-section units.
uinp_energy = nue.UnfoldInput.build(
    var         = var_energy,
    reco_df     = mcmc_signal_df,
    true_df     = mcnue_sig_df,
    syst_output = signal_outputs_total['energy'],
)
uinp_direct = nue.UnfoldInput.build(
    var         = var_direction,
    reco_df     = mcmc_signal_df,
    true_df     = mcnue_sig_df,
    syst_output = signal_outputs_total['direction'],
)

# GENIE + MCstat: the covariance used for the fake data tests.
# cov_ms_ms is the post-constraint total-rate covariance; pass it as total_cov
# to unfold() so that the CCBC-corrected covariance is used directly.
energy_fdt_cov = nue.get_ccbc_cov(
    Bs_output    = signal_outputs_bkg['energy'],
    Ps_output    = signal_outputs_sig['energy'],
    nc_output    = sideband_outputs['energy'],
    allowed_keys = ['MCstat', 'GENIE'],
)
direct_fdt_cov = nue.get_ccbc_cov(
    Bs_output    = signal_outputs_bkg['direction'],
    Ps_output    = signal_outputs_sig['direction'],
    nc_output    = sideband_outputs['direction'],
    allowed_keys = ['MCstat', 'GENIE'],
)

In [ ]:
# FDT: scale QE (mode==0) events up by 20% in signal and sideband.
# make_fake_data_hists uses the CCBC constraint to correct the background
# prediction using the modified sideband observation (fd_nc).
fd_meas, fd_true, fd_ns, fd_nc, fd_Bs = nue.make_fake_data_hists(
    reco_df   = mcmc_signal_df,
    true_df   = mcnue_sig_df,
    side_df   = mcmc_sideband_df,
    mcbnb_pot = mcbnb_pot,
    var       = var_energy,
    ccbc_cov  = energy_fdt_cov,
    reco_mask = mcmc_signal_df.slc.truth.genie_mode == 0,
    true_mask = mcnue_sig_df.genie_mode == 0,
    side_mask = mcmc_sideband_df.slc.truth.genie_mode == 0,
    weight    = 1.2,
)

# Show pre/post-constraint total rate vs the fake data observation
fig, axes, _ = nue.plot_ccbc_fd_comparison(
    energy_fdt_cov, fd_nc, fd_ns, fd_Bs,
    ns_hist    = signal_outputs_total['energy'].rate_hist_cv,
    var_config = var_energy,
)
plt.suptitle("Scale QE by +20%")
plt.show()

res = uinp_energy.unfold(
    WienerSVD,
    measure   = fd_meas,
    total_cov = energy_fdt_cov['cov_ms_ms'],
)
nue.plot_unfolded_result(
    result       = res,
    var          = var_energy,
    truths       = {"baseline": uinp_energy.cv_signal, "modified": fd_true},
    truth_colors = {"baseline": "navy", "modified": "seagreen"},
)
plt.title("Scale QE by +20% (with CCBC)")
plt.show()

In [ ]:
# FDT: scale DIS (mode==2) events up by 100% in signal and sideband.
fd_meas, fd_true, fd_ns, fd_nc, fd_Bs = nue.make_fake_data_hists(
    reco_df   = mcmc_signal_df,
    true_df   = mcnue_sig_df,
    side_df   = mcmc_sideband_df,
    mcbnb_pot = mcbnb_pot,
    var       = var_direction,
    ccbc_cov  = direct_fdt_cov,
    reco_mask = mcmc_signal_df.slc.truth.genie_mode == 2,
    true_mask = mcnue_sig_df.genie_mode == 2,
    side_mask = mcmc_sideband_df.slc.truth.genie_mode == 2,
    weight    = 2.0,
)

fig, axes, _ = nue.plot_ccbc_fd_comparison(
    direct_fdt_cov, fd_nc, fd_ns, fd_Bs,
    ns_hist    = signal_outputs_total['direction'].rate_hist_cv,
    var_config = var_direction,
)
plt.suptitle("Scale DIS by +100%")
plt.show()

res = uinp_direct.unfold(
    WienerSVD,
    measure   = fd_meas,
    total_cov = direct_fdt_cov['cov_ms_ms'],
)
nue.plot_unfolded_result(
    result       = res,
    var          = var_direction,
    truths       = {"baseline": uinp_direct.cv_signal, "modified": fd_true},
    truth_colors = {"baseline": "navy", "modified": "seagreen"},
)
plt.title("Scale DIS by +100% (with CCBC)")
plt.show()

In [ ]:
# FDT: scale primary pi0 production down by 50%.
# npi0 > 0 selects events with at least one primary neutral pion.
fd_meas, fd_true, fd_ns, fd_nc, fd_Bs = nue.make_fake_data_hists(
    reco_df   = mcmc_signal_df,
    true_df   = mcnue_sig_df,
    side_df   = mcmc_sideband_df,
    mcbnb_pot = mcbnb_pot,
    var       = var_direction,
    ccbc_cov  = direct_fdt_cov,
    reco_mask = mcmc_signal_df.slc.truth.npi0 > 0,
    true_mask = mcnue_sig_df.npi0 > 0,
    side_mask = mcmc_sideband_df.slc.truth.npi0 > 0,
    weight    = 0.5,
)

fig, axes, _ = nue.plot_ccbc_fd_comparison(
    direct_fdt_cov, fd_nc, fd_ns, fd_Bs,
    ns_hist    = signal_outputs_total['direction'].rate_hist_cv,
    var_config = var_direction,
)
plt.suptitle(r"Scale primary $\pi^0$ production by $-50\%$")
plt.show()

res = uinp_direct.unfold(
    WienerSVD,
    measure   = fd_meas,
    total_cov = direct_fdt_cov['cov_ms_ms'],
)
nue.plot_unfolded_result(
    result       = res,
    var          = var_direction,
    truths       = {"baseline": uinp_direct.cv_signal, "modified": fd_true},
    truth_colors = {"baseline": "navy", "modified": "seagreen"},
)
plt.title(r"Scale primary $\pi^0$ production by $-50\%$ (with CCBC)")
plt.show()

## Alternate-generator test (GiBUU)

In [ ]:
# GiBUU is a fully independent nuclear transport model used as a fake dataset
# to probe model-dependence and to test the CCBC constraint with a realistic
# alternate-physics sideband observation.
gibuu_df, gibuu_pot, _ = nue.load_mc(
    dfs_dir + "mc_gibuu.df",
    keys=['nuecc', 'hdr', 'histpotdf'],
    cuts=nue.DEFAULT_CUTS,
)
gibuu_sb_df, _, _ = nue.load_mc(
    dfs_dir + "mc_gibuu.df",
    keys=['nuecc', 'hdr', 'histpotdf'],
    cuts=nue.SIDEBAND_CUTS,
)
gibuu_sig_df = nue.load_dfs(dfs_dir + "mc_gibuu.df", ['mcnuecc'], n_max_concat=np.inf)['mcnuecc']
gibuu_sig_df = nue.define_signal(gibuu_sig_df)
gibuu_sig_df = gibuu_sig_df[gibuu_sig_df.signal == 0]

# CCBC requires all histograms and covariances in consistent units (events at mcbnb_pot).
# Scale GiBUU raw counts by mcbnb_pot / gibuu_pot to match the nominal MC normalization.
gibuu_pot_scale = mcbnb_pot / gibuu_pot
print(f"GiBUU signal-region events passing selection: {len(gibuu_df)}")
print(f"GiBUU sideband events passing selection:      {len(gibuu_sb_df)}")

In [ ]:
def _hist(df, col, bins):
    return nue.get_hist1d(data=nue.ensure_lexsorted(df, axis=1)[col], bins=bins)

for var_cfg, unf_cfg, syst_bkg_out, sideband_out in [
    (var_energy,    uinp_energy, signal_outputs_bkg['energy'],    sideband_outputs['energy']),
    (var_direction, uinp_direct, signal_outputs_bkg['direction'], sideband_outputs['direction']),
]:
    vn = var_cfg.var_save_name

    # Raw GiBUU event counts; scale to mcbnb_pot for unit consistency with syst outputs.
    fd_total_raw = _hist(gibuu_df,     var_cfg.var_evt_reco_col, var_cfg.bins)
    fd_nc_raw    = _hist(gibuu_sb_df,  var_cfg.var_evt_reco_col, var_cfg.bins)
    fd_true_raw  = _hist(gibuu_sig_df, var_cfg.var_nu_col,       var_cfg.bins)

    fd_total = fd_total_raw * gibuu_pot_scale  # events at mcbnb_pot
    fd_nc    = fd_nc_raw    * gibuu_pot_scale
    fd_true  = fd_true_raw  * gibuu_pot_scale

    # Poisson data-stat covariance (events² at mcbnb_pot).
    # Passing these to get_ccbc_cov softens the constraint when the sideband
    # or signal-region observations are statistically noisy.
    data_stat_nc = np.diag(fd_nc_raw)    * gibuu_pot_scale ** 2
    data_stat_ns = np.diag(fd_total_raw) * gibuu_pot_scale ** 2

    ccbc_cov = nue.get_ccbc_cov(
        Bs_output    = signal_outputs_bkg[vn],
        Ps_output    = signal_outputs_sig[vn],
        nc_output    = sideband_out,
        allowed_keys = ['MCstat', 'GENIE'],
        data_stat_nc = data_stat_nc,
        data_stat_ns = data_stat_ns,
    )

    # Apply the CCBC linear predictor to get the constrained background estimate
    fd_Bs_constr   = nue.get_constrained_background(ccbc_cov, fd_nc)
    fd_meas_uncon  = fd_total - syst_bkg_out.rate_hist_cv
    fd_meas_constr = fd_total - fd_Bs_constr

    res_uncon  = unf_cfg.unfold(WienerSVD, measure=fd_meas_uncon,  total_cov=ccbc_cov['cov_ns_ns'])
    res_constr = unf_cfg.unfold(WienerSVD, measure=fd_meas_constr, total_cov=ccbc_cov['cov_ms_ms'])

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, title, res, tc in [
        (axes[0], "Unconstrained", res_uncon,  {'baseline (GENIE)': 'C0',   'GiBUU truth': 'yellowgreen'}),
        (axes[1], "Constrained",   res_constr, {'baseline (GENIE)': 'navy', 'GiBUU truth': 'seagreen'}),
    ]:
        nue.plot_unfolded_result(
            result       = res,
            var          = var_cfg,
            ax           = ax,
            truths       = {"baseline (GENIE)": unf_cfg.cv_signal, "GiBUU truth": fd_true},
            data_label   = "unfolded GiBUU",
            truth_colors = tc,
        )
        ax.set_title(title)
    plt.suptitle(f"GiBUU alternate-generator test ({var_cfg.var_plot_name})")
    plt.tight_layout()
    plt.show()